# Code for Bradley-Terry

In [17]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm
from scipy.special import expit
from scipy.sparse.linalg import eigs
import pandas as pd

def measure_loglikelihood(matrix, probs):
    """
    return loglikelihood
    see equation before (2) in https://en.wikipedia.org/wiki/Bradley%E2%80%93Terry_model
    """
    m = matrix * np.log(probs[:, None] / (probs[:, None] + probs[None, :]))
    logprob = np.sum(m)
    return logprob


class ZermeloBradleyTerry:
    """
    Zermelo formula for solving standard Bradley-Terry
    see equation 3 in https://en.wikipedia.org/wiki/Bradley%E2%80%93Terry_model
    """

    def __init__(self, n_iters=100):
        self.n_iters = n_iters

    def fit(self, W):
        """
        W : array of shape (n_players, n_players)
            W[i,j] = number of times i beat j
        """
        n = W.shape[0]
        probs = np.ones(n) / n
        logliks = []
        for _ in range(self.n_iters):
            denom = (W + W.T) / (probs[:, None] + probs[None, :])
            probs_new = np.sum(W, axis=1) / np.sum(denom, axis=1)
            probs = probs_new / np.prod(probs_new) ** (1 / n)
            logliks.append(measure_loglikelihood(W, probs))
        ranking = np.argsort(-probs)
        return {'weights': probs, 'loglikelihoods_history': logliks, 'ranking': ranking}


class BayesianBradleyTerry:
    """
    Bayesian Bradley-Terry model for pairwise comparisons.
    https://www.jmlr.org/papers/volume24/22-0907/22-0907.pdf

    Parameters:
    -----------
    n_players : int
        Number of players/items to rank
    n_iter : int
        Number of MCMC iterations
    warmup : int
        Number of warmup iterations to discard
    random_seed : int
        Random seed for reproducibility
    """

    def __init__(self, n_players, n_iter=2000, warmup=1000, random_seed=42):
        self.n_players = n_players
        self.n_iter = n_iter
        self.warmup = warmup
        self.random_seed = random_seed
        np.random.seed(random_seed)
        self.log_liks = []

    def win_probability(self, beta_i, beta_j):
        """Probability that player i beats player j"""
        return 1 / (1 + np.exp(-(beta_i - beta_j)))

    def log_likelihood(self, beta, W, N):
        """Log-likelihood of the data given beta parameters"""
        log_lik = 0
        t = len(beta)

        for i in range(t):
            for j in range(i + 1, t):
                if N[i, j] > 0:
                    p_ij = self.win_probability(beta[i], beta[j])
                    # Binomial log-likelihood for W[i,j] wins out of N[i,j] trials
                    log_lik += W[i, j] * np.log(p_ij) + W[j, i] * np.log(1 - p_ij)
                    # Add log binomial coefficient (constant w.r.t. parameters)
                    # log_lik += sp.loggamma(N[i, j] + 1) - sp.loggamma(W[i, j] + 1) - sp.loggamma(W[j, i] + 1)
        self.log_liks.append(log_lik)
        return log_lik

    def log_prior(self, beta, sigma):
        """Log-prior for beta and sigma"""
        # Beta ~ Normal(0, sigma)
        beta_prior = -0.5 * np.sum(beta ** 2) / (sigma ** 2) - 0.5 * self.n_players * np.log(sigma)

        # Sigma ~ LogNormal(0, 0.5)
        sigma_prior = -0.5 * (np.log(sigma) / 0.5) ** 2 - np.log(sigma)  # np.log(0.5 * sigma * np.sqrt(2*np.pi))

        return beta_prior + sigma_prior

    def log_posterior(self, beta, sigma, W, N):
        """Log-posterior (unnormalized)"""
        return self.log_likelihood(beta, W, N) + self.log_prior(beta, sigma)

    def sample_posterior(self, W, N):
        """
        Sample from posterior using Metropolis-Hastings.

        Parameters:
        -----------
        W : array of shape (n_players, n_players)
            W[i,j] = number of times i beat j
        N : array of shape (n_players, n_players)
            N[i,j] = total number of matches between i and j

        Returns:
        --------
        beta_samples : array of shape (n_iter - warmup, n_players)
            MCMC samples of beta parameters
        sigma_samples : array of shape (n_iter - warmup,)
            MCMC samples of sigma parameter
        """
        t = self.n_players

        # Initialize parameters
        beta = np.random.randn(t) * 0.1
        sigma = 1.0

        # Normalize beta to sum to 0 (identifiability constraint)
        beta = beta - np.mean(beta)

        # Store samples
        beta_samples = np.zeros((self.n_iter - self.warmup, t))
        sigma_samples = np.zeros(self.n_iter - self.warmup)

        # Proposal standard deviations
        beta_proposal_sd = 0.1
        sigma_proposal_sd = 0.1

        current_log_post = self.log_posterior(beta, sigma, W, N)

        for it in range(self.n_iter):
            # Sample new beta
            beta_proposed = beta + np.random.randn(t) * beta_proposal_sd
            beta_proposed = beta_proposed - np.mean(beta_proposed)  # enforce sum=0

            # Sample new sigma
            sigma_proposed = np.exp(np.log(sigma) + np.random.randn() * sigma_proposal_sd)

            # Calculate proposed log-posterior
            proposed_log_post = self.log_posterior(beta_proposed, sigma_proposed, W, N)

            # Metropolis-Hastings acceptance ratio
            log_accept_ratio = proposed_log_post - current_log_post

            # For sigma: add Jacobian term for log-transform
            log_accept_ratio += np.log(sigma_proposed) - np.log(sigma)

            # Accept or reject
            if np.log(np.random.rand()) < log_accept_ratio:
                beta = beta_proposed
                sigma = sigma_proposed
                current_log_post = proposed_log_post

            # Store sample after warmup
            if it >= self.warmup:
                idx = it - self.warmup
                beta_samples[idx] = beta
                sigma_samples[idx] = sigma

        return beta_samples, sigma_samples

    def fit(self, W, N):
        """
        Fit the Bayesian Bradley-Terry model.

        Returns:
        --------
        result : dict
            Contains:
            - 'beta_mean': posterior mean of beta
            - 'beta_std': posterior std of beta
            - 'w_mean': posterior mean of w = exp(beta)
            - 'w_std': posterior std of w
            - 'sigma_mean': posterior mean of sigma
            - 'samples': all MCMC samples
        """
        beta_samples, sigma_samples = self.sample_posterior(W, N)

        # Compute posterior means
        beta_mean = np.mean(beta_samples, axis=0)
        beta_std = np.std(beta_samples, axis=0)

        # Convert to w
        w_samples = np.exp(beta_samples)
        # w_samples = w_samples / np.sum(w_samples, axis=1, keepdims=True) #(normalize to sum to 1)

        w_mean = np.exp(beta_mean)  # np.mean(w_samples, axis=0)
        # w_std = np.std(w_samples, axis=0)

        sigma_mean = np.mean(sigma_samples)

        # Compute ranking
        ranking = np.argsort(-beta_mean)  # descending order

        # confidence interval
        intervals = []
        for i in range(len(beta_mean)):
            std = np.std(beta_samples[:, i])
            intervals.append(norm.interval(0.95, loc=beta_mean[i],
                                scale=std / (beta_samples.shape[1] ** 0.5)))
        intervals = np.array(intervals)
        
        intervals_beta = []
        for i in range(len(beta_mean)):
            for j in range(i + 1, len(beta_mean)):
                std = np.std(beta_samples[:, i] - beta_samples[:, j])
                intervals_beta.append(norm.interval(0.95, loc=beta_mean[i] - beta_mean[j],
                                               scale=std / (beta_samples.shape[1] ** 0.5)))
        intervals_beta = np.array(intervals_beta)

        return {
            'beta_mean': beta_mean,
            'beta_std': beta_std,
            'weights': w_mean,
            # 'w_std': w_std,
            'sigma_mean': sigma_mean,
            'samples': {
                'beta': beta_samples,
                'sigma': sigma_samples,
                'w': w_samples
            },
            'ranking': ranking,
            'intervals_beta':intervals,
            'intervals_w': np.exp(intervals),
            'prob_intervals_beta': intervals_beta,
            'prob_intervals_w': expit(intervals_beta)
        }


class SpokoinyBradleyTerry:
    """
    Solves model from https://arxiv.org/pdf/2503.15045

    v_G = argmax_v (L(v) - ||Gv||^2/2)

    Parameters
    ----------
    S : numpy array of shape (n, n)
        Matrix where S[j,m] = sum of Y_{jm}^{(ℓ)} (number of times j beats m)
        Only upper triangular part of S is used.
    N : numpy array of shape (n, n)
        Matrix where N[j,m] = number of comparisons between j and m
        N should be symmetric with zeros on diagonal
    G : numpy array of shape (n, n) - adjacency matrix of graph
    tol : float
        Tolerance for convergence
    max_iter : int
        Maximum number of iterations

    Returns
    -------
    v_opt : numpy array of length n
        Optimal v that maximizes penalized log-likelihood
    """

    def __init__(self, S, N, G, tol=1e-8, max_iter=1000):
        self.n = S.shape[0]
        self.S = S
        self.N = N
        self.G_squared = G @ G.T
        self.v = np.ones(self.n) / self.n
        self.tol = tol
        self.max_iter = max_iter

    def neg_log_likelihood_penalized(self, v):
        """
        Compute -[L(v) - 0.5 * ||Gv||^2]
        """
        L = 0.0
        for m in range(self.n):
            for j in range(m):
                if self.N[j, m] > 0:
                    diff = v[j] - v[m]
                    L += diff * self.S[j, m] - self.N[j, m] * np.log(1 + np.exp(diff))
        penalty = 0.5 * v @ self.G_squared @ v
        return -L + penalty

    def grad_neg_log_likelihood_penalized(self, v):
        """
        Compute gradient of -[L(v) - 0.5 * ||Gv||^2]
        """
        grad = np.zeros(self.n)
        for m in range(self.n):
            for j in range(m):
                if j != m and self.N[min(j, m), max(j, m)] > 0:
                    diff = v[j] - v[m]
                    g = self.S[j, m] - self.N[j, m] * np.exp(diff) / (1 + np.exp(diff))
                    grad[j] += g
                    grad[m] -= g
        grad_penalty = self.G_squared @ v
        return -grad + grad_penalty

    def fit(self):
        result = minimize(
            self.neg_log_likelihood_penalized,
            self.v,
            jac=self.grad_neg_log_likelihood_penalized,
            method='L-BFGS-B',
            options={'gtol': self.tol, 'maxiter': self.max_iter, 'disp': False}
        )

        v_opt = result.x
        # Apply identifiability constraint: v_1 = 0
        # This doesn't change the model predictions but makes solution unique
        self.v = v_opt - v_opt[0]
        # norm weights such that prod(weights)=1
        weights = np.exp(v_opt - np.mean(v_opt))
        ranking = np.argsort(-v_opt)
        return {'weights': weights,  'ranking': ranking}
    
    
class RankCentralityBT:
    """
    https://arxiv.org/pdf/2110.03874
    """
    def __init__(self, tol=1e-7, max_iter=1000):
        self.max_iter = max_iter
        self.tol = tol

    def fit(self, Y, A=None, d=None):
        """
        Rank Centrality algorithm for Bradley-Terry model estimation.

        Parameters:
        -----------
        Y : np.ndarray, shape (n, n)
            Matrix where Y[i,j] = number of times i beat j in comparisons.
            Diagonal entries should be zero.

        A : np.ndarray, shape (n, n), optional
            Adjacency matrix of comparison graph (1 if i and j compared, else 0).
            If None, computed from Y (A[i,j] = 1 if Y[i,j] + Y[j,i] > 0).

        max_iter : int, default=1000
            Maximum number of power iterations for stationary distribution.

        tol : float, default=1e-10
            Convergence tolerance for stationary distribution.

        d : float, optional
            Scaling parameter. If None, calculated as 2 * n * p where
            p is the edge density of the comparison graph.

        Returns:
        --------
        theta_hat : np.ndarray, shape (n,)
            Estimated skill parameters (centered to sum to 0).

        pi_hat : np.ndarray, shape (n,)
            Stationary distribution of the Markov chain.

        P : np.ndarray, shape (n, n)
            Transition matrix of the Markov chain.
        """
        n = Y.shape[0]

        # Create adjacency matrix if not provided
        if A is None:
            A = ((Y + Y.T) > 0).astype(float)

        # Compute edge density p
        m = np.sum(A) / 2  # number of edges (undirected)
        p = m / (n * (n - 1) / 2)  # edge density in complete graph

        # Set scaling parameter d
        if d is None:
            d = 2 * n * p

        # Compute normalized comparison frequencies
        total_comparisons = Y + Y.T
        # Avoid division by zero
        with np.errstate(divide='ignore', invalid='ignore'):
            bar_y = np.where(total_comparisons > 0, Y / total_comparisons, 0)

        # Initialize transition matrix P
        P = np.zeros((n, n))

        # Fill off-diagonal entries
        for i in range(n):
            for j in range(n):
                if i != j and A[i, j] > 0:
                    P[i, j] = (1/d) * A[i, j] * bar_y[j, i]

        # Fill diagonal entries to make each row sum to 1
        row_sums = P.sum(axis=1)
        np.fill_diagonal(P, 1 - row_sums)

        # Ensure non-negative entries (clip small negative values due to numerical issues)
        P = np.maximum(P, 0)
        # Renormalize rows to sum to 1
        P = P / P.sum(axis=1, keepdims=True)

        # Compute stationary distribution pi_hat using power iteration
        '''
        pi_hat = np.ones(n)  # uniform initial distribution
        pi_hat /= np.linalg.norm(pi_hat)

        for _ in range(max_iter):
            pi_next = pi_hat @ P
            pi_next /= np.linalg.norm(pi_next)

            # Check convergence
            if np.linalg.norm(pi_next - pi_hat) < tol:
                pi_hat = pi_next
                break

            pi_hat = pi_next

        # Normalize stationary distribution
        pi_hat = pi_hat / pi_hat.sum()
        '''
        _, pi_hat = eigs(P.T, k=1)
        pi_hat = pi_hat.T[0].real
        pi_hat /= np.sum(pi_hat)

        # Compute theta_hat from pi_hat
        log_pi = np.log(pi_hat)
        # Center to satisfy identifiability condition sum(theta) = 0
        theta_hat = log_pi - np.mean(log_pi)

        return {'weights': np.exp(theta_hat), 
                'theta_hat': theta_hat,
                'pi_hat': pi_hat,
                'ranking': np.argsort(-pi_hat),
               }


In [18]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm, expon, gamma
from scipy.special import expit
from tqdm import tqdm

class PlackettLuce:
    """
    Plackett-Luce model for ranking data with EM and Gibbs sampling.

    Parameters:
    -----------
    K : int
        Number of items
    a : float, default=1
        Gamma prior shape parameter for λ
    b : float, default=0
        Gamma prior scale parameter for λ (1/rate)
    """

    def __init__(self, K, a=1, b=0):
        self.K = K  # number of items
        self.a = a  # Gamma prior shape
        self.b = b  # Gamma prior scale (1/rate)

    def _create_delta(self, rankings):
        """
        Create delta indicators for rankings.

        Parameters:
        -----------
        rankings : list of lists
            Each inner list contains ranking where first element is top ranked.

        Returns:
        --------
        delta : list of lists
            delta[i][j][k] = 1 if item k is in position j or worse in ranking i
        w : array
            w[k] = number of rankings where item k is not last
        """
        n = len(rankings)
        delta = []
        w = np.zeros(self.K)

        for i, rank_i in enumerate(rankings):
            p_i = len(rank_i)
            delta_i = []

            for j in range(p_i - 1):
                delta_ij = np.zeros(self.K)
                # Items from position j to end (j is 0-indexed)
                items_in_set = rank_i[j:]
                delta_ij[items_in_set] = 1
                delta_i.append(delta_ij)

                # Update w for items not in last position
                if j < p_i - 1:  # If not the last item in this ranking
                    w[rank_i[j]] += 1

            delta.append(delta_i)

        return delta, w

    def fit_em(self, rankings, max_iter=1000, tol=1e-6, init_lambda=None, verbose=False):
        """
        Fit Plackett-Luce model using EM algorithm.

        Parameters:
        -----------
        rankings : list of lists
            Rankings data
        max_iter : int, default=1000
            Maximum number of iterations
        tol : float, default=1e-6
            Convergence tolerance
        init_lambda : array, optional
            Initial λ values
        verbose : bool, default=False
            Whether to print progress

        Returns:
        --------
        lambda_hat : array
            Estimated λ parameters
        history : dict
            Convergence history
        """
        n = len(rankings)
        delta, w = self._create_delta(rankings)

        # Initialize λ
        if init_lambda is None:
            lambda_curr = np.ones(self.K) / self.K
        else:
            lambda_curr = init_lambda.copy()

        history = {'lambda': [], 'log_lik': []}

        for t in range(max_iter):
            lambda_new = np.zeros(self.K)

            # EM update for each item k
            for k in range(self.K):
                numerator = self.a - 1 + w[k]
                denominator = self.b

                for i in range(n):
                    rank_i = rankings[i]
                    p_i = len(rank_i)

                    for j in range(p_i - 1):
                        # Sum of λ for items from position j to end
                        items_in_set = rank_i[j:]
                        sum_lambda = lambda_curr[items_in_set].sum()

                        # Check if item k is in the set
                        if k in items_in_set:
                            denominator += 1 / sum_lambda

                lambda_new[k] = numerator / denominator if denominator > 0 else 0

            # Normalize (optional but helps with identifiability)
            lambda_new = lambda_new / lambda_new.sum()

            # Calculate log-likelihood
            log_lik = self._log_likelihood(rankings, lambda_new)
            history['lambda'].append(lambda_new.copy())
            history['log_lik'].append(log_lik)

            # Check convergence
            diff = np.max(np.abs(lambda_new - lambda_curr))

            if verbose and t % 100 == 0:
                print(f"Iteration {t}: max diff = {diff:.6f}, log-lik = {log_lik:.4f}")

            if diff < tol:
                if verbose:
                    print(f"EM converged after {t + 1} iterations")
                break

            lambda_curr = lambda_new

        return {'weights':lambda_curr, 
                'history':history,
                'ranking':np.argsort(-lambda_curr)
               }

    def fit_gibbs(self, rankings, n_iter=5000, burn_in=1000, init_lambda=None):
        """
        Gibbs sampling for Plackett-Luce model.

        Parameters:
        -----------
        rankings : list of lists
            Rankings data
        n_iter : int, default=5000
            Total number of MCMC iterations
        burn_in : int, default=1000
            Number of burn-in iterations
        init_lambda : array, optional
            Initial λ values

        Returns:
        --------
        samples : dict
            Dictionary containing λ samples and latent Z samples
        """
        n = len(rankings)
        delta, w = self._create_delta(rankings)

        # Initialize
        if init_lambda is None:
            lambda_curr = np.ones(self.K) / self.K
        else:
            lambda_curr = init_lambda.copy()

        # Storage
        lambda_samples = np.zeros((n_iter - burn_in, self.K))
        z_samples = []

        # Precompute arrays for speed
        ranking_arrays = [np.array(rank, dtype=np.int32) for rank in rankings]
        ranking_lengths = np.array([len(rank) for rank in rankings], dtype=np.int32)
        pos_arrays = []
        for rank_array in ranking_arrays:
            pos = -np.ones(self.K, dtype=np.int32)
            for idx, item in enumerate(rank_array):
                pos[item] = idx
            pos_arrays.append(pos)
        pos_arrays = np.stack(pos_arrays, axis=0)

        for t in tqdm(range(n_iter), desc="Gibbs Sampling"):
            # Step 1: Sample latent variables Z and prefix sums
            Z = []
            prefix_sums = []
            for i in range(n):
                rank_i = ranking_arrays[i]
                p_i = ranking_lengths[i]
                if p_i > 1:
                    rate = np.flip(np.flip(lambda_curr[rank_i]).cumsum())[:p_i-1]
                    Z_i = np.random.exponential(scale=1.0 / rate)
                    Z.append(Z_i)
                    cumsum = np.cumsum(Z_i)
                    prefix_full = np.zeros(p_i, dtype=float)
                    prefix_full[:p_i-1] = cumsum
                    prefix_full[p_i-1] = cumsum[-1]
                    prefix_sums.append(prefix_full)
                else:
                    Z.append(np.array([], dtype=float))
                    prefix_sums.append(np.array([0.0], dtype=float))

            # Step 2: Sample λ parameters (vectorized)
            shape = self.a + w
            rate_term = np.full(self.K, self.b, dtype=float)
            for i in range(n):
                pos = pos_arrays[i]
                mask = pos >= 0
                if not np.any(mask):
                    continue
                rate_term[mask] += prefix_sums[i][pos[mask]]

            lambda_curr = np.random.gamma(shape=shape, scale=1.0 / rate_term)
            lambda_curr = lambda_curr / lambda_curr.sum()

            # Store samples after burn-in
            if t >= burn_in:
                lambda_samples[t - burn_in] = lambda_curr
                z_samples.append(Z)
            
        lambda_mean = lambda_samples.mean(axis=0)

        return {
            'lambda': lambda_samples,
            'weights': lambda_mean,
            'z': z_samples,
            'delta': delta,
            'w': w,
            'ranking': np.argsort(-lambda_mean)
        }

    def _log_likelihood(self, rankings, lambda_vec):
        """
        Calculate log-likelihood for given rankings and λ.
        """
        log_lik = 0

        for rank_i in rankings:
            p_i = len(rank_i)

            for j in range(p_i - 1):
                # Numerator: λ of item at position j
                numerator = lambda_vec[rank_i[j]]

                # Denominator: sum of λ for items from position j to end
                items_in_set = rank_i[j:]
                denominator = lambda_vec[items_in_set].sum()

                log_lik += np.log(numerator) - np.log(denominator)

        return log_lik

    def predict_probability(self, lambda_vec, items):
        """
        Predict probability of ranking for a set of items.

        Parameters:
        -----------
        lambda_vec : array
            λ parameters
        items : list
            List of item indices

        Returns:
        --------
        prob : float
            Probability of this ranking
        """
        prob = 1.0
        remaining_items = list(items)

        for pos, item in enumerate(items):
            if pos == len(items) - 1:
                break

            prob *= lambda_vec[item] / sum(lambda_vec[i] for i in remaining_items)
            remaining_items.remove(item)

        return prob
    
    

def make_rankings_pl(df, datasets):
    algs = pd.DataFrame({'alg':list(df.columns), 'n': range(len(df.columns))}).set_index('alg')
    rankings = []
    for dataset in datasets:
        values = df.loc[dataset].values
        valid_mask = ~np.isnan(values)
        if not valid_mask.any():
            continue
        valid_indices = np.where(valid_mask)[0]
        sorted_indices = valid_indices[np.argsort(-values[valid_mask])]
        alg_names = df.columns[sorted_indices].tolist()
        rankings.append(algs.loc[alg_names].values.T[0])
    return rankings


# Section 5.3

In [19]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os
import matplotlib.colors as mcolors

MODEL_NAME_MAP = {
    "LightGCN_Uniform": "LightGCN",
}

def normalize_model_name(name: str) -> str:
    if name in MODEL_NAME_MAP:
        return MODEL_NAME_MAP[name]
    return name

path = os.path.join("reports", "best_trials_raw.csv")
df_raw = pd.read_csv(path)

df_raw["model"] = df_raw["model"].apply(normalize_model_name)

allowed_datasets = [
    'Amazon2014-Amazon-Instant-Video',
    'Amazon2014-Apps-For-Android',
    'Amazon2014-Automotive',
    'Amazon2014-Baby',
    'Amazon2014-Beauty',
    'Amazon2014-CDs-And-Vinyl',
    'Amazon2014-Cell-Phones-And-Accessories',
    'Amazon2014-Clothing-Shoes-And-Jewelry',
    'Amazon2014-Digital-Music',
    'Amazon2014-Grocery-And-Gourmet',
    'Amazon2014-Health-And-Personal-Care',
    'Amazon2014-Home-And-Kitchen',
    'Amazon2014-Kindle-Store',
    'Amazon2014-Musical-Instruments',
    'Amazon2014-Office-Products',
    'Amazon2014-Patio-Lawn-And-Garden',
    'Amazon2014-Pet-Supplies',
    'Amazon2014-Sports-And-Outdoors',
    'Amazon2014-Tools-And-Home-Improvement',
    'Amazon2014-Toys-And-Games',
    'Amazon2014-Video-Games',
    'Amazon2018-Arts-Crafts-And-Sewing',
    'Amazon2018-Automotive',
    'Amazon2018-Cell-Phones-And-Accessories',
    'Amazon2018-Digital-Music',
    'Amazon2018-Gift-Cards',
    'Amazon2018-Grocery-And-Gourmet-Food',
    'Amazon2018-Industrial-And-Scientific',
    'Amazon2018-Luxury-Beauty',
    'Amazon2018-Magazine-Subscriptions',
    'Amazon2018-Musical-Instruments',
    'Amazon2018-Office-Products',
    'Amazon2018-Patio-Lawn-And-Garden',
    'Amazon2018-Prime-Pantry',
    'Amazon2018-Software',
    'Amazon2018-Sports-And-Outdoors',
    'Amazon2018-Video-Games',
    'Behance',
    'BookCrossing',
    'CiaoDVD',
    'CiteULike-A',
    'CiteULike-T',
    'DeliveryHero-SE',
    'FilmTrust',
    'FoodComRecipes',
    'Foursquare-NYC1',
    'Foursquare-NYC2',
    'Foursquare-Tokyo',
    'Frappe',
    'GoogleLocal2018',
    'GoogleLocal2021-Alaska',
    'GoogleLocal2021-Delaware',
    'GoogleLocal2021-District-Of-Columbia',
    'GoogleLocal2021-New-Hampshire',
    'GoogleLocal2021-North-Dakota',
    'GoogleLocal2021-Rhode-Island',
    'GoogleLocal2021-South-Dakota',
    'GoogleLocal2021-Vermont',
    'GoogleLocal2021-Wyoming',
    'Hetrec-LastFM',
    'Jester-4',
    'KGRec-Music',
    'LearningFromSets',
    'Librarything',
    'MarketBias-ModCloth',
    'ModCloth-Clothing-Fit',
    'MovieLens-100K',
    'MovieLens-1M',
    'MovieLens-Latest-Small',
    'MovieTweetings',
    'Myket-Android',
    'Personality',
    'RentTheRunway',
    'Retailrocket',
    'Steam-Australian-Reviews',
    'WikiLens',
    'Yoochoose',
    'Amazon2014-Books',
    'Amazon2014-Electronics',
    'Amazon2014-Movies-And-TV',
    'Amazon2018-CDs-And-Vinyl',
    'Amazon2018-Electronics',
    'Amazon2018-Home-And-Kitchen',
    'Amazon2018-Kindle-Store',
    'Amazon2018-Movies-And-TV',
    'Amazon2018-Pet-Supplies',
    'Amazon2018-Tools-And-Home-Improvement',
    'Amazon2018-Toys-And-Games',
    'Anime-Recommendations-Database',
]
if len(allowed_datasets) > 0:
    df_raw = df_raw[df_raw["dataset"].isin(allowed_datasets)].copy()

tex_output_path = os.path.join("plots", "rankings_tables.tex")

datasets_info_path = os.path.join("reports", "datasets_info.csv")
datasets_info = pd.read_csv(datasets_info_path)
datasets_info = datasets_info.rename(columns={'Датасет':'Dataset'})

sns.set_theme(style="whitegrid", context="paper", font_scale=1.4)

colors = {
    'BTL': '#0072B2',
    'draw': '#56B4E9',
    'mean': '#D55E00',
    'sum': '#009E73'
}

pretty_names = {
    'popular_random': 'PopRandom',
    'user_knn': 'User-KNN',
    'item_knn': 'Item-KNN',
    'seq_knn': 'Seq-KNN',
    'MF_SGD': 'SGD MF',
    'bpr_mf': 'BPR',
    'als_implicit': 'ALS',
    'pure_svd': 'PureSVD',
    'ease_r': 'EASEr',
    'GASATF': 'GASATF',
    'LightGCN': 'LightGCN',
    'ultragcn': 'UltraGCN',
    'SASRec': 'SASRec',
    'Random': 'Random',
    'random': 'Random',
    'sasrec': 'SASRec'          
}


In [20]:
metric_for_missing = "ndcg@10"
col = f"test/{metric_for_missing}"
pivot = df_raw.pivot_table(index="dataset", columns="model", values=col, aggfunc="first")

datasets_info_path = os.path.join("reports", "datasets_info.csv")
datasets_info_local = pd.read_csv(datasets_info_path)

dataset_col = None
for cand in ["Dataset", "dataset", "Датасет"]:
    if cand in datasets_info_local.columns:
        dataset_col = cand
        break
if dataset_col is None:
    raise ValueError(f"Dataset column not found in datasets_info.csv. Columns: {list(datasets_info_local.columns)}")

if dataset_col != "Dataset":
    datasets_info_local = datasets_info_local.rename(columns={dataset_col: "Dataset"})

idx_col = None
for c in datasets_info_local.columns:
    if str(c).lower() == "dataset_idx":
        idx_col = c
        break
if idx_col is None:
    raise ValueError(f"dataset_idx column not found in datasets_info.csv. Columns: {list(datasets_info_local.columns)}")

dataset_to_idx = datasets_info_local.set_index("Dataset")[idx_col].to_dict()

missing_per_model = {}
missing_idx_per_model = {}

for model in pivot.columns:
    missing_datasets = pivot.index[pivot[model].isna()].tolist()
    missing_per_model[model] = missing_datasets
    missing_idx = [dataset_to_idx.get(ds) for ds in missing_datasets]
    missing_idx_per_model[model] = missing_idx

for model in pivot.columns:
    print(f"\nModel: {model}")
    print("Missing datasets:", missing_per_model[model])
    print("Missing dataset_idx:", missing_idx_per_model[model])


Model: GASATF
Missing datasets: ['Amazon2018-Movies-And-TV']
Missing dataset_idx: [86]

Model: LightGCN
Missing datasets: ['Amazon2014-Books', 'Amazon2018-Electronics']
Missing dataset_idx: [89, 90]

Model: MF_SGD
Missing datasets: []
Missing dataset_idx: []

Model: als_implicit
Missing datasets: []
Missing dataset_idx: []

Model: bpr_mf
Missing datasets: ['Amazon2014-Books', 'Amazon2018-Electronics', 'Amazon2018-Home-And-Kitchen']
Missing dataset_idx: [89, 90, 91]

Model: ease_r
Missing datasets: ['Amazon2014-Books', 'Amazon2014-CDs-And-Vinyl', 'Amazon2014-Electronics', 'Amazon2014-Kindle-Store', 'Amazon2014-Movies-And-TV', 'Amazon2018-Automotive', 'Amazon2018-CDs-And-Vinyl', 'Amazon2018-Cell-Phones-And-Accessories', 'Amazon2018-Electronics', 'Amazon2018-Grocery-And-Gourmet-Food', 'Amazon2018-Home-And-Kitchen', 'Amazon2018-Kindle-Store', 'Amazon2018-Movies-And-TV', 'Amazon2018-Pet-Supplies', 'Amazon2018-Sports-And-Outdoors', 'Amazon2018-Tools-And-Home-Improvement', 'Amazon2018-Toys-A

In [21]:
def build_metric_tables(metrics_list):
    tables = {}
    for metric in metrics_list:
        col = f"test/{metric}"
        tables[metric] = df_raw.pivot_table(index="dataset", columns="model", values=col, aggfunc="first")
    return tables

def make_tables(df_, datasets):
    n = len(algorithms)
    table = np.zeros((n, n))
    for dataset in datasets:
        mean_row = df_.loc[dataset]
        for i in range(n):
            for j in range(i+1, n):
                alg_i = algorithms[i]
                alg_j = algorithms[j]
                m1 = mean_row[alg_i]
                m2 = mean_row[alg_j]
                if pd.isna(m1) or pd.isna(m2):
                    continue
                if m1 > m2:
                    table[i, j] += 1
                elif m1 == m2:
                    table[i, j] += 0.5
                    table[j, i] += 0.5
                else:
                    table[j, i] += 1
    return table

def plot_heatmap(weights, dataset_group_name, out_pdf, algs=None):
    w = np.asarray(weights, dtype=float)
    idx = np.argsort(-w)
    local_algs = algorithms if algs is None else algs
    sorted_algs = np.asarray(local_algs)[idx]
    sorted_pretty = [pretty_names.get(a, a) for a in sorted_algs]
    w = w[idx]

    P = w[:, None] / (w[:, None] + w[None, :])
    np.fill_diagonal(P, 0.5)

    dfP = pd.DataFrame(P, index=sorted_pretty, columns=sorted_pretty)

    fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)
    sns.heatmap(dfP, ax=ax, cmap="vlag", vmin=0, vmax=1, center=0.5,
                square=True, linewidths=0.25, linecolor="white",
                cbar=False)

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=60, labelsize=16)
    ax.tick_params(axis="y", rotation=0, labelsize=16)
    plt.savefig(out_pdf, format="pdf", bbox_inches="tight")
    plt.close()


def save_colorbar(out_pdf, orientation="vertical"):
    """Save the heatmap colorbar as a standalone PDF."""
    import matplotlib.cm as cm
    cmap = sns.color_palette("vlag", as_cmap=True)
    norm_obj = mcolors.TwoSlopeNorm(vmin=0, vcenter=0.5, vmax=1)
    sm = cm.ScalarMappable(cmap=cmap, norm=norm_obj)
    sm.set_array([])

    if orientation == "vertical":
        fig, ax = plt.subplots(figsize=(0.5, 4))
        cb = fig.colorbar(sm, cax=ax, orientation="vertical")
        cb.set_label("Win Probability", fontsize=10)
    else:
        fig, ax = plt.subplots(figsize=(4, 0.5))
        cb = fig.colorbar(sm, cax=ax, orientation="horizontal")
        cb.set_label("Win Probability", fontsize=10)

    cb.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
    cb.ax.tick_params(labelsize=8)
    plt.savefig(out_pdf, format="pdf", bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)


def fit_plackett_luce_rankings(
    df_mean,
    datasets,
    a=1,
    b=0,
    max_iter=1000,
    tol=1e-8,
    n_iter=2000,
    burn_in=1000,
    verbose=False,
 ):
    rankings = make_rankings_pl(df_mean, datasets)
    K = len(df_mean.columns)
    algs = list(df_mean.columns)

    counts = np.zeros(K, dtype=int)
    for ranking in rankings:
        for idx in ranking:
            counts[int(idx)] += 1
    missing = [algs[i] for i in range(K) if counts[i] == 0]
    if len(missing) > 0:
        print(f"PL: no comparisons for {len(missing)} algorithms: {missing}")

    model = PlackettLuce(K=K, a=a, b=b)
    results_em = model.fit_em(rankings, max_iter=max_iter, tol=tol, verbose=verbose)

    model = PlackettLuce(K=K, a=a, b=b)
    results_gibbs = model.fit_gibbs(
        rankings, n_iter=n_iter, burn_in=burn_in, init_lambda=results_em['weights']
    )

    return results_em, results_gibbs

def save_ranking(
    bt,
    mn,
    sm,
    dataset_type,
    metric_name,
    tex_path,
    pl_em=None,
 ):
    def _ranks(x):
        """Rank 1 = largest value. Single argsort."""
        order = np.argsort(-x)
        ranks = np.empty_like(order)
        ranks[order] = np.arange(1, len(x) + 1)
        return ranks

    r_btl = _ranks(bt)
    r_sum = _ranks(sm)
    r_mean = _ranks(mn)

    delta_sum = r_sum - r_btl
    delta_mean = r_mean - r_btl

    r_pl_em = None
    delta_pl_em = None

    if pl_em is not None:
        r_pl_em = _ranks(pl_em)
        delta_pl_em = r_pl_em - r_btl
    
    def fmt_rank_with_arrow(rank, delta):
        if delta == 0:
            return f"{int(rank)} (0)"
        elif delta > 0:
            return f"{int(rank)} ($\\mathord{{\\uparrow}}{abs(delta)}$)"
        else:
            return f"{int(rank)} ($\\mathord{{\\downarrow}}{abs(delta)}$)"

    algo_col = [pretty_names.get(a, a) for a in algorithms]

    rank_columns = [
        ("BTL", r_btl, None),
        ("Sum ($\Delta$)", r_sum, delta_sum),
        ("Mean ($\Delta$)", r_mean, delta_mean),
    ]
    if r_pl_em is not None:
        rank_columns.append(("PL-EM ($\Delta$)", r_pl_em, delta_pl_em))

    data = {
        ('Algorithms', ''): algo_col,
    }
    for label, ranks, deltas in rank_columns:
        if deltas is None:
            data[("Rank", label)] = ranks
        else:
            data[("Rank", label)] = [
                fmt_rank_with_arrow(ranks[i], deltas[i])
                for i in range(len(algorithms))
            ]

    df_rank = pd.DataFrame(data)
    df_rank.columns = pd.MultiIndex.from_tuples(df_rank.columns)
    df_rank = df_rank.sort_values(('Rank', 'BTL'))

    n_rank_cols = len(rank_columns)
    column_format = "l|" + "c" * n_rank_cols

    latex_body = df_rank.to_latex(
        index=False,
        column_format=column_format,
        multicolumn=True,
        multicolumn_format="c",
        escape=False
    )

    latex_body = latex_body.replace('\\toprule', '').replace('\\midrule', '').replace('\\bottomrule', '')

    header_part = rf"& \\multicolumn{{{n_rank_cols}}}{{c}}{{Rank}} \\\\"
    if header_part in latex_body:
        latex_body = latex_body.replace(header_part, header_part + f" \\cline{{2-{n_rank_cols + 1}}}")

    content = latex_body \
        .replace(r'\begin{tabular}{%s}' % column_format, '') \
        .replace(r'\end{tabular}', '') \
        .strip()

    cols_row = "& " + " & ".join([label for label, _, _ in rank_columns]) + " \\\\"
    content = content.replace(cols_row, cols_row + " \\hline")

    caption_text = (
        f"Ranking of recommendation algorithms on {dataset_type} datasets "
        f"by {metric_name}"
    )

    label_name = f"rank_{dataset_type}_{metric_name}".replace(' ', '_')
    label_name = label_name.replace('/', '_')

    latex_code = f"""
\\begin{{table}}[t]
\\centering
\\renewcommand{{\\arraystretch}}{{1.2}} 
\\caption{{{caption_text}}}
\\label{{tab:{label_name}}}
\\begin{{tabular}}{{{column_format}}}
\\hline
{content}
\\hline
\\end{{tabular}}
\\end{{table}}
"""

    with open(tex_path, "a", encoding="utf-8") as f:
        f.write(latex_code)
        f.write("\n\n")

In [22]:
import pandas as pd

def compute_kendall_tables(metrics_list, plot_configs, tex_path):
    keep_groups = {
        'all': 'All',
        'sequential': 'Sequential',
        'unsequential': 'Orderless',
        'top20_Density': 'Dense',
        'worst20_Density': 'Sparse',
        'top20_User-Item_Ratio': 'Mostly users',
        'worst20_User-Item_Ratio': 'Mostly items',
        'top20_Mean_Interaction_per_User': 'Long history',
        'worst20_Mean_Interaction_per_User': 'Short history'
    }
    rows = []

    for metric in metrics_list:
        df_mean = df_raw.pivot_table(index="dataset",
                                     columns="model", values=f"test/{metric}", aggfunc="first")
        algs = list(df_mean.columns)

        for datasets_sel, name in plot_configs:
            if name not in keep_groups:
                continue

            datasets_sel = [d for d in datasets_sel if d in df_mean.index]
            if len(datasets_sel) == 0:
                continue

            W = make_tables(df_mean, datasets_sel)
            N = W + W.T
            results = BayesianBradleyTerry(n_players=W.shape[0]).fit(W, N)

            w_btl = results['weights']
            w_mean = df_mean.loc[datasets_sel][algs].mean().reindex(algs).values
            w_sum = df_mean.loc[datasets_sel][algs].sum().reindex(algs).values

            results_em, _ = fit_plackett_luce_rankings(
                df_mean, datasets_sel, verbose=False
            )
            w_pl = results_em['weights']

            valid_mask = np.isfinite(w_mean) & np.isfinite(w_sum) & np.isfinite(w_pl)
            if valid_mask.sum() < 2:
                tau_mean = 0.0
                tau_sum = 0.0
                tau_pl = 0.0
            else:
                v_btl = w_btl[valid_mask]
                v_mean = w_mean[valid_mask]
                v_sum = w_sum[valid_mask]
                v_pl = w_pl[valid_mask]

                def safe_kendall(r1, r2):
                    r1 = np.argsort(-r1)
                    r2 = np.argsort(-r2)

                    m = len(r1)
                    x = np.zeros(m)
                    y = np.zeros(m)
                    for i in range(m):
                        x[r1[i]] = i
                        y[r2[i]] = i
                    conc = 0
                    disc = 0
                    for i in range(m):
                        for j in range(i+1, m):
                            if (x[i]>=x[j] and y[i]>=y[j]) or (x[i]<=x[j] and y[i]<=y[j]):
                                conc+=1
                            else:
                                disc +=1 
                    return 2*(conc - disc) / (m*(m-1))

                tau_mean = safe_kendall(v_btl, v_mean)
                tau_sum = safe_kendall(v_btl, v_sum)
                tau_pl = safe_kendall(v_btl, v_pl)

            rows.append({
                "Dataset group": keep_groups[name],
                "Metric": metric,
                "Mean": tau_mean,
                "Sum": tau_sum,
                "PL": tau_pl
            })

    df_res = pd.DataFrame(rows)
    wide = df_res.pivot(index="Dataset group", columns="Metric", values=["Mean", "Sum", "PL"])
    wide = wide.swaplevel(0, 1, axis=1)
    wide = wide.reindex(columns=metrics_list, level=0)
    wide = wide.sort_index(axis=1, level=[0, 1], ascending=[True, True])

    num_metrics = len(metrics_list)
    
    column_spec = "l" + "ccc" * num_metrics

    formatted_metrics = [m.replace('_', '\\_') for m in metrics_list]
    
    top_header_parts = [""]
    cmidrules = []
    
    start_col = 2
    for m in formatted_metrics:
        top_header_parts.append(f"\\multicolumn{{3}}{{c}}{{{m}}}")
        cmidrules.append(f"\\cmidrule(lr){{{start_col}-{start_col+2}}}")
        start_col += 3
        
    top_header = " & ".join(top_header_parts) + " \\\\"
    cmidrule_str = " ".join(cmidrules)

    sub_header_cells = ["Mean & Sum & PL"] * num_metrics
    sub_header = "Dataset group & " + " & ".join(sub_header_cells) + " \\\\"

    content_rows = []
    ordered_groups = []
    seen_groups = set()
    for _, name in plot_configs:
        if name not in keep_groups:
            continue
        display = keep_groups[name]
        if display not in seen_groups:
            ordered_groups.append(display)
            seen_groups.add(display)

    for group_name in ordered_groups:
        if group_name not in wide.index:
            continue

        row_str = [group_name]
        for metric in metrics_list:
            val_mean = wide.loc[group_name, (metric, 'Mean')]
            val_sum = wide.loc[group_name, (metric, 'Sum')]
            val_pl = wide.loc[group_name, (metric, 'PL')]

            row_str.append(f"{val_mean:.3f}")
            row_str.append(f"{val_sum:.3f}")
            row_str.append(f"{val_pl:.3f}")
        
        content_rows.append(" & ".join(row_str) + " \\\\")

    content = "\n".join(content_rows)

    latex_code = f"""
\\begin{{table}}[t]
\\centering
\\caption{{Kendall correlations between BT and baselines}}
\\label{{tab:kendall_btl_baselines}}
\\resizebox{{\\linewidth}}{{!}}{{
\\begin{{tabular}}{{{column_spec}}}
\\toprule
{top_header}
{cmidrule_str}
{sub_header}
\\midrule
{content}
\\bottomrule
\\end{{tabular}}
}}
\\end{{table}}
"""
    with open(tex_path, "a", encoding="utf-8") as f:
        f.write(latex_code)
        f.write("\n\n")

    print(f"Table appended to {tex_path}")


In [23]:
def filter_musor(df_):
    if 'Remove?' not in df_.columns:
        return df_.copy()
    return df_[df_['Remove?'] != 'Yes'].copy()

def get_strong_sequential_datasets(df_):
    if 'Sequential?' not in df_.columns:
        return df_.copy()
    return df_[df_['Sequential?'] == 'Yes'].copy()

def get_strong_unsequential_datasets(df_):
    if 'Not Sequential?' not in df_.columns:
        return df_.copy()
    return df_[df_['Not Sequential?'] == 'Yes'].copy()

def get_dataset_names(df_subset):
    if 'Dataset' in df_subset.columns:
        return df_subset['Dataset'].tolist()
    return list(df_subset.index)


datasets_info_clean = filter_musor(datasets_info.copy())

TOPN = 20
sequential_datasets = get_strong_sequential_datasets(datasets_info_clean)
unsequential_datasets = get_strong_unsequential_datasets(datasets_info_clean)

def _top20_bottom20(df_, metric_col, top_n):
    if metric_col not in df_.columns:
        return [], []
    top_ds = get_dataset_names(df_.nlargest(top_n, metric_col))
    bottom_ds = get_dataset_names(df_.nsmallest(top_n, metric_col))
    return top_ds, bottom_ds

dense_ds, sparse_ds = _top20_bottom20(datasets_info_clean, 'Density', TOPN)
users_heavy_ds, items_heavy_ds = _top20_bottom20(datasets_info_clean, 'User-Item Ratio', TOPN)
long_hist_ds, short_hist_ds = _top20_bottom20(datasets_info_clean, 'Mean Interaction per User', TOPN)

plot_configs = [
    (get_dataset_names(datasets_info_clean), 'all'),
    (get_dataset_names(sequential_datasets), 'sequential'),
    (get_dataset_names(unsequential_datasets), 'unsequential'),
    (dense_ds, 'top20_Density'),
    (sparse_ds, 'worst20_Density'),
    (users_heavy_ds, 'top20_User-Item_Ratio'),
    (items_heavy_ds, 'worst20_User-Item_Ratio'),
    (long_hist_ds, 'top20_Mean_Interaction_per_User'),
    (short_hist_ds, 'worst20_Mean_Interaction_per_User')
]

contrast_pairs = []

# Dense vs Sparse (Density: higher = denser)
if dense_ds and sparse_ds:
    contrast_pairs.append({
        "title": "Density",
        "left_label": "Dense",
        "right_label": "Sparse",
        "left_ds": dense_ds,
        "right_ds": sparse_ds,
        "label": "density_dense_vs_sparse"
    })

# Mostly users vs Mostly items (User-Item Ratio: higher = more users than items)
if users_heavy_ds and items_heavy_ds:
    contrast_pairs.append({
        "title": "User-Item Ratio",
        "left_label": "Mostly users",
        "right_label": "Mostly items",
        "left_ds": users_heavy_ds,
        "right_ds": items_heavy_ds,
        "label": "user_item_ratio_users_vs_items"
    })

# Short vs Long average user history (Mean Interaction per User: higher = longer history)
if long_hist_ds and short_hist_ds:
    contrast_pairs.append({
        "title": "Mean Interaction per User",
        "left_label": "Long history",
        "right_label": "Short history",
        "left_ds": long_hist_ds,
        "right_ds": short_hist_ds,
        "label": "mean_interaction_per_user_long_vs_short"
    })

# Sequential vs Orderless
seq_ds = get_dataset_names(sequential_datasets)
unseq_ds = get_dataset_names(unsequential_datasets)
if seq_ds and unseq_ds:
    contrast_pairs.append({
        "title": "Sequentiality",
        "left_label": "Sequential",
        "right_label": "Orderless",
        "left_ds": seq_ds,
        "right_ds": unseq_ds,
        "label": "sequential_vs_orderless"
    })

plot_config_map = {name: datasets_sel for datasets_sel, name in plot_configs}

In [24]:
os.makedirs('plots', exist_ok=True)

save_colorbar('plots/heatmap_colorbar_v.pdf', orientation='vertical')
save_colorbar('plots/heatmap_colorbar_h.pdf', orientation='horizontal')

def btl_ranking_for_group(datasets_sel, df_mean_local):
    datasets_sel = [d for d in datasets_sel if d in df_mean_local.index]
    if len(datasets_sel) == 0:
        return []
    W = make_tables(df_mean_local, datasets_sel)
    N = W + W.T
    results = BayesianBradleyTerry(n_players=W.shape[0]).fit(W, N)
    w_btl = results['weights']
    order = np.argsort(-w_btl)
    return [pretty_names.get(algorithms[i], algorithms[i]) for i in order]

def _latex_escape(text: str) -> str:
    return text.replace('_', '\\_')

def build_contrast_tables_wide(
    pairs,
    metric_name,
    df_mean_local,
    tex_path,
    top_k=100000,
    pairs_per_table=4,
 ):
    metric_tag = metric_name.replace('/', '_').replace('@', 'at')
    prepared = []
    
    
    for pair in pairs:
        left_rank = btl_ranking_for_group(pair["left_ds"], df_mean_local)
        right_rank = btl_ranking_for_group(pair["right_ds"], df_mean_local)
        
        left_rank = [pretty_names.get(x, x) for x in left_rank]
        right_rank = [pretty_names.get(x, x) for x in right_rank]
        
        k = min(top_k, len(left_rank), len(right_rank))
        if k == 0:
            continue
        prepared.append({
            "title": _latex_escape(pair["title"]),
            "left_label": _latex_escape(pair["left_label"]),
            "right_label": _latex_escape(pair["right_label"]),
            "left_rank": left_rank,
            "right_rank": right_rank,
            "k": k
        })

    if len(prepared) == 0:
        return

    def _chunks(lst, size):
        for i in range(0, len(lst), size):
            yield lst[i:i + size]

    for table_idx, chunk in enumerate(_chunks(prepared, pairs_per_table), start=1):
        k = min(item["k"] for item in chunk)
        n_pairs = len(chunk)
        
        col_spec = "c" + "l" * (2 * n_pairs) 
        
        top_header_parts = [""]
        cmidrules = []
        current_col = 2
        for item in chunk:
            top_header_parts.append(f"\\multicolumn{{2}}{{c}}{{{item['title']}}}")
            cmidrules.append(f"\\cmidrule(lr){{{current_col}-{current_col+1}}}")
            current_col += 2
            
        top_header = " & ".join(top_header_parts) + " \\\\"
        cmidrule_str = " ".join(cmidrules)

        sub_header_cells = []
        for item in chunk:
            sub_header_cells.append(f"\\multicolumn{{1}}{{c}}{{{item['left_label']}}}")
            sub_header_cells.append(f"\\multicolumn{{1}}{{c}}{{{item['right_label']}}}")
            
        sub_header = "Rank & " + " & ".join(sub_header_cells) + " \\\\"
        
        rows = []
        for i in range(k):
            cells = []
            for item in chunk:
                cells.append(item["left_rank"][i])
                cells.append(item["right_rank"][i])
            rows.append(f"{i + 1} & " + " & ".join(cells) + " \\\\")
            
        body = "\n".join(rows)
        
        caption = f"BTL rankings for dataset contrasts (metric: {metric_name})"
        label = f"tab:contrast_pairs_{metric_tag}_{table_idx}"
        
        latex_code = f"""
\\begin{{table*}}[t]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
\\setlength{{\\tabcolsep}}{{3pt}}
\\resizebox{{\\textwidth}}{{!}}{{
\\begin{{tabular}}{{{col_spec}}}
\\toprule
{top_header}
{cmidrule_str}
{sub_header}
\\midrule
{body}
\\bottomrule
\\end{{tabular}}
}}
\\end{{table*}}
"""
        with open(tex_path, "a", encoding="utf-8") as f:
            f.write(latex_code)
            f.write("\n\n")

        print(f"Appended wide contrast table {table_idx} to {tex_path}")



with open(tex_output_path, "w", encoding="utf-8") as f:
    f.write("")

keep_groups = {
        'all': 'All',
        'sequential': 'Sequential',
        'unsequential': 'Orderless',
        'top20_Density': 'Dense',
        'worst20_Density': 'Sparse',
        'top20_User-Item_Ratio': 'Mostly users',
        'worst20_User-Item_Ratio': 'Mostly items',
        'top20_Mean_Interaction_per_User': 'Long history',
        'worst20_Mean_Interaction_per_User': 'Short history'
    }

metrics = ['ndcg@10']
for metric in metrics:
    if metric != "ndcg@10":
        continue
    metric_tag = metric.replace('/', '_').replace('@', 'at')
    df_mean = df_raw.pivot_table(index="dataset", columns="model", values=f"test/{metric}", aggfunc="first")
    algorithms = list(df_mean.columns)

    for datasets_sel, name in plot_configs:
        datasets_sel = [d for d in datasets_sel if d in df_mean.index]
        if len(datasets_sel) == 0:
            continue


        W = make_tables(df_mean, datasets_sel)
        N = W + W.T
        results = BayesianBradleyTerry(n_players=W.shape[0]).fit(W, N)
        w_btl = results['weights']
        w_samples = results['samples']['w']

        plot_heatmap(
            w_btl,
            keep_groups[name],
            f'plots/btl_heatmap_{name}_{metric_tag}.pdf',
        )

        save_colorbar(f'plots/btl_heatmap_colorbar_{metric_tag}.pdf', orientation='vertical')

        w_mean = df_mean.loc[datasets_sel][algorithms].mean().reindex(algorithms).values
        w_sum = df_mean.loc[datasets_sel][algorithms].sum().reindex(algorithms).values

        results_em, results_gibbs = fit_plackett_luce_rankings(
            df_mean, datasets_sel, verbose=False
        )
        w_pl_em = results_em['weights']
        w_pl_gibbs = results_gibbs['weights']
        w_pl_samples = results_gibbs['lambda']

        save_ranking(
            w_btl,
            w_mean,
            w_sum,
            dataset_type=name,
            metric_name=metric,
            tex_path=tex_output_path,
            pl_em=w_pl_em,
        )

    build_contrast_tables_wide(
        contrast_pairs,
        metric,
        df_mean,
        tex_output_path,
        top_k=100000,
        pairs_per_table=4,
    )


Gibbs Sampling: 100%|██████████| 2000/2000 [00:00<00:00, 2958.00it/s]


Appended wide contrast table 1 to plots/rankings_tables.tex


In [25]:
kendall_metrics = ['ndcg@10', 'recall@10', 'coverage@10']
compute_kendall_tables(kendall_metrics, plot_configs, tex_output_path)

Gibbs Sampling: 100%|██████████| 2000/2000 [00:00<00:00, 2984.13it/s]

Table appended to plots/rankings_tables.tex
